# Aleph — a language model that is one matrix

Aleph spends its parameters on computation rather than on a lookup table. A
conventional tiny language model at `V=4096`, `N=512` pays `2·V·N = 4.2M`
parameters for an embedding table and an output head, against `N² = 262k` for
the layer that does the work: 94% of the "model" is a dictionary, and any
parameter-efficiency figure quoted against that total is measuring the
dictionary.

Here the token interface is **frozen**. Each token gets a fixed sparse ternary
code drawn from a seeded generator, and both directions read the same table:

    inject(t) = H[t] · in_gain
    logits    = h @ Hᵀ · out_scale + bias

That costs `V + N + 1` trainable parameters instead of `2·V·N` — 4,609 against
4,194,304 at `V=4096, N=512`, **910× less** — and what it asks in exchange is
that the core learn to emit points in code space.

**Why a random code separates.** Codes are near-orthogonal by
Johnson–Lindenstrauss: the typical inner product between two of them is
`N^(-1/2)` (0.044 at `N=512`, measured 0.0442 at every density), so `V` enters
only logarithmically and a wider vocabulary is nearly free.

**What this notebook produces.** A trained model, a bits-per-byte score on
held-out text, the parameter report broken out by role, and the matched-seed
ablation arms that turn a number into a claim.


## 1 — Environment

Prints what we actually got, since Colab's assignment varies and every budget below depends on it.


In [ ]:
import subprocess, sys, os, time, json, math, pathlib

print("--- GPU ---")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "no GPU — set Runtime > Change runtime type > GPU")

import torch
print(f"\ntorch {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"  {p.name}  {p.total_memory / 2**30:.1f} GiB  sm_{p.major}{p.minor}  {p.multi_processor_count} SMs")
    print(f"  bf16 supported: {torch.cuda.is_bf16_supported()}")
    # A recurrent core multiplies the same matrix every step, so fp16's narrow
    # exponent is the wrong trade. bf16 where available, fp32 otherwise.
    torch.set_float32_matmul_precision('high')

IN_COLAB = 'google.colab' in sys.modules
print(f"\ncolab: {IN_COLAB}")


Clone and install. `datasets` and `tokenizers` are the only additions; torch is already present on Colab.


In [ ]:
REPO = "https://github.com/theomgdev/aleph"
if IN_COLAB:
    if not pathlib.Path('aleph-repo').exists():
        # A local copy works too: upload the repo and skip this cell.
        r = subprocess.run(["git", "clone", "--depth", "1", REPO, "aleph-repo"],
                           capture_output=True, text=True)
        print(r.stdout or r.stderr)
    if pathlib.Path('aleph-repo').exists():
        os.chdir('aleph-repo')
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "datasets", "tokenizers"], check=False)

sys.path.insert(0, os.getcwd())
from aleph import Codebook, AlephCore, Trainer, __version__
print(f"aleph {__version__} from {os.getcwd()}")


Mount Drive so a checkpoint survives the session. Colab disconnects on
idle and on a 12/24-hour ceiling; a run that cannot resume loses everything it
had. Skip this cell to keep checkpoints session-local.


In [ ]:
CKPT_DIR = 'ckpt'
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CKPT_DIR = '/content/drive/MyDrive/aleph_ckpt'
        os.makedirs(CKPT_DIR, exist_ok=True)
        print(f"checkpoints -> {CKPT_DIR}")
    except Exception as e:
        print(f"Drive unavailable ({type(e).__name__}); checkpoints stay session-local")
os.makedirs(CKPT_DIR, exist_ok=True)


## 2 — Corpus

TinyStories (Eldan & Li, [arXiv:2305.07759](https://arxiv.org/abs/2305.07759)):
stories restricted to the vocabulary and syntax of a young child, which is the
corpus that makes a few-million-parameter model worth measuring at all. On web
text the same model produces mush, and the score would describe the corpus's
difficulty rather than the architecture.

The BPE fit and the encode together take a few minutes and are cached, so
re-running this cell is free. `--stories` is the knob: 400k is ~130M tokens,
enough that a sub-1M-parameter model is not the thing that runs out.


In [ ]:
STORIES = 400_000     # ~130M tokens
VOCAB   = 4096        # nearly free under the codebook, so chosen for sequence length

t0 = time.perf_counter()
r = subprocess.run([sys.executable, "-u", "tools/build_corpus.py",
                    "--tokenizer", "bpe", "--vocab", str(VOCAB),
                    "--stories", str(STORIES)],
                   capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode:
    print("STDERR:", r.stderr[-2000:])
print(f"\n[{time.perf_counter() - t0:.0f}s]")


In [ ]:
from tools.experiment import find_cache
meta = find_cache('data')
print(f"corpus   {pathlib.Path(meta['path']).name}")
print(f"  train  {meta['train_tokens']:,} tokens")
print(f"  val    {meta['val_tokens']:,} tokens")
print(f"  V      {meta['vocab_size']}   {meta['bytes_per_token']:.2f} chars/token")
# bits/byte divides by the text the tokens encode, so this figure is part of
# every score below: a comparison against another cache is not a comparison.


## 3 — Smoke test

Asserts wiring, not learning. At this budget neither the loss nor the accuracy
moves measurably, so a gate on either would read the seed; what it checks
instead is deterministic — that the loss reaches every parameter family, that
the frozen code table does not move, and that `W`'s diagonal stays zero.

Run it before anything long. It fails in seconds where a real run fails in
hours.


In [ ]:
r = subprocess.run([sys.executable, "-u", "tools/experiment.py", "--mode", "smoke",
                    "--neurons", "128", "--depth", "4"],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode:
    print("STDERR:", r.stderr[-3000:])
    raise SystemExit("smoke failed — fix before training")


## 4 — The run

Three facts measured on an RTX 3060 Ti set these defaults, and they are worth
knowing before changing them:

**The loop is launch-bound.** 128× the batch cost 1.15× the time — 1,042 tok/s
at batch 1 against 115,851 at batch 128. Batch is therefore nearly free and
should be as large as memory allows; it is the first thing to raise on a bigger
GPU.

**`depth` saturates in time but not in capacity.** Branches are concatenated
into one wide matmul pair, so depth 4, 8 and 16 all cost ~151–158 ms while
parameters go 531k → 793k → 1.32M. Past depth 4 capacity is genuinely free,
which is why the default is 8.

**Warmup is not optional.** `depth=4` reached 0.0192 nats at lr 1e-3 and
diverged at 3e-3, where `depth=0` tolerated both. The extra gradient path
narrows the usable rate, so a depth comparison without warmup measures the
schedule.


In [ ]:
import torch
free = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0

CFG = dict(
    neurons = 512,     # core is N^2; the matrix IS the model
    depth   = 8,       # free in time past 4, so take the capacity
    batch   = 256 if free > 30 else (128 if free > 14 else 32),
    seq     = 256,     # also the truncated-BPTT window: credit spans a chunk
    steps   = 6000,
    lr      = 1e-3,
    warmup  = 300,
)
print(f"{free:.0f} GiB -> batch {CFG['batch']}")
print(f"budget: {CFG['steps'] * CFG['batch'] * CFG['seq'] / 1e6:.0f}M tokens"
      f"  ({CFG['steps'] * CFG['batch'] * CFG['seq'] / meta['train_tokens']:.1f} epochs)")
# Epochs is the number that tells data starvation from over-parameterization:
# a corpus crossed dozens of times is starved, not too large a model.


The log line carries what distinguishes this architecture's failure
modes: `bpb` is the tokenizer-independent score, `ep` is epochs (starvation vs
over-parameterization), `|g|` and `clip%` are the gradient and how often the
clip binds, and `|h|` must sit at `√N` — 22.6 at `N=512` — because per-step
RMSNorm pins it there. A run that goes wrong goes wrong in one of those
columns.


In [ ]:
args = [sys.executable, "-u", "tools/experiment.py", "--mode", "train",
        "--neurons", str(CFG['neurons']), "--depth", str(CFG['depth']),
        "--batch", str(CFG['batch']), "--seq", str(CFG['seq']),
        "--steps", str(CFG['steps']), "--lr", str(CFG['lr']),
        "--warmup", str(CFG['warmup']), "--ckpt", CKPT_DIR,
        "--tag", "main", "--sample", "--resume"]
print(" ".join(args), "\n")

# Streamed rather than captured: a multi-hour cell that prints nothing is
# indistinguishable from one that has hung.
proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f"\nexit {proc.returncode}")


## 5 — Ablations

A mechanism becomes a claim here, or does not. Each arm runs at the same seed
with one axis changed, and a negative result is reported as a negative result:
going looking for the configuration that rescues a mechanism is how an
efficiency claim stops being true.

`depth` first — the capacity that was free in wall-clock.


In [ ]:
r = subprocess.run([sys.executable, "-u", "tools/experiment.py", "--mode", "ablate",
                    "--ablate", "depth=0,4,8,16", "--steps", "1200",
                    "--neurons", str(CFG['neurons']), "--batch", str(CFG['batch']),
                    "--seq", str(CFG['seq']), "--lr", str(CFG['lr'])],
                   capture_output=True, text=True)
print(r.stdout[-3000:])


`nnz` — code density. The prediction is that it does not matter: the
typical inner product is `N^(-1/2)` at every density and only the worst-case
tail depends on `nnz`. If quality is flat here, the codebook's separability
argument holds.


In [ ]:
r = subprocess.run([sys.executable, "-u", "tools/experiment.py", "--mode", "ablate",
                    "--ablate", "nnz=8,32,64,256", "--steps", "1200",
                    "--neurons", str(CFG['neurons']), "--depth", str(CFG['depth']),
                    "--batch", str(CFG['batch']), "--seq", str(CFG['seq'])],
                   capture_output=True, text=True)
print(r.stdout[-3000:])


`neurons` — the parameter-efficiency curve itself. The core is `N²`, so
this is the axis a record claim is made along, and the interesting quantity is
bits per byte against trainable parameters rather than either alone.


In [ ]:
r = subprocess.run([sys.executable, "-u", "tools/experiment.py", "--mode", "ablate",
                    "--ablate", "neurons=128,256,512,768", "--steps", "1500",
                    "--depth", str(CFG['depth']), "--batch", str(CFG['batch']),
                    "--seq", str(CFG['seq'])],
                   capture_output=True, text=True)
print(r.stdout[-3000:])


## 6 — Samples, and the honest accounting

Generation is the only check that reads as language rather than as a number.
A model can score well and produce nothing: read the samples.


In [ ]:
r = subprocess.run([sys.executable, "-u", "tools/experiment.py", "--mode", "eval",
                    "--ckpt", CKPT_DIR, "--tag", "main", "--sample"],
                   capture_output=True, text=True)
print(r.stdout[-4000:])


### What may and may not be claimed

The split matters more than the total, so the report prints four roles.

**Fair to say.** The *core* — the trained computation — is `N² − N` parameters,
and the token interface costs `V + N + 1` against the `2·V·N` a learned
embedding and head would cost. At `N=512, V=4096` that is 4,609 against
4,194,304. Quote bits per byte beside the trainable total and the cache it was
measured on.

**Not fair to say.** That the frozen table is free in the information-theoretic
sense — it is `V·nnz` fixed values, chosen by a seed rather than by training,
and a reader is entitled to count the seed's `V × N` support however they like.
State it as *trainable* parameters, which is what it is.

**Also not fair.** Comparing a bits-per-byte figure against one from a
different tokenizer or a rebuilt cache. The metric is tokenizer-independent but
it is defined against the held-out split it was computed on, so quote the
corpus's train/val token counts with any comparison — including between two of
our own runs.


In [ ]:
from aleph import Codebook, AlephCore
cb = Codebook(meta['vocab_size'], CFG['neurons'], device='cuda' if torch.cuda.is_available() else 'cpu')
m = AlephCore(CFG['neurons'], cb, depth=CFG['depth'])
r = m.param_report()
conv = 2 * meta['vocab_size'] * CFG['neurons']

print(f"{'role':<12} {'params':>12}")
for k in ('core', 'refine', 'interface', 'modulation'):
    print(f"{k:<12} {r[k]:>12,}")
print(f"{'TOTAL':<12} {r['total']:>12,}  trainable")
print(f"\ninterface  {r['interface']:,} vs {conv:,} for a learned embed+head"
      f"   ({conv / r['interface']:.0f}x less)")
print(f"frozen     {cb.H.numel():,} table values, generated from seed {cb.seed:#x}")
print(f"           ({(cb.H != 0).sum().item():,} nonzero)")
